In [38]:
%pip install holidays

Note: you may need to restart the kernel to use updated packages.


In [39]:
%cd Proyecto_Big_Data

[Errno 2] No such file or directory: 'Proyecto_Big_Data'
/home/jovyan/work/Proyecto_Big_Data


In [40]:
import os
import subprocess
import pandas as pd
import requests
from pathlib import Path
from datetime import datetime
import re

# --- CONFIGURACIÓN DE RUTAS ---
UNRAR_EXE = '/usr/bin/unrar' 
FOLDER_BRONCE = "./datos_clima_bronce"
FOLDER_EXTRACT = "./datos_clima_temp"
BASE_URL = "https://datosclima.es/capturadatos/"

Path(FOLDER_BRONCE).mkdir(parents=True, exist_ok=True)
Path(FOLDER_EXTRACT).mkdir(parents=True, exist_ok=True)

# --- FUNCIONES DE LIMPIEZA Y APOYO ---

def limpiar_valor(v):
    """
    Limpia strings tipo '19.5 (14:40)' y los convierte en floats (19.5).
    Si no hay número, devuelve None.
    """
    if pd.isna(v) or v == '' or v == 'nan': 
        return None
    # Buscamos el primer número decimal o entero en el texto
    match = re.search(r"[-+]?\d*\.\d+|\d+", str(v))
    return float(match.group()) if match else None

def extraer_fecha_del_nombre_archivo(nombre_f, anio_contexto, mes_contexto):
    """
    Extrae el día del nombre del archivo (ej: '05.xls') y monta la fecha completa.
    """
    numeros = re.findall(r'\d+', nombre_f)
    dia = "01"
    if numeros:
        # El primer número suele ser el día en estos archivos
        posible_dia = numeros[0].zfill(2)
        if int(posible_dia) <= 31:
            dia = posible_dia
            
    return f"{anio_contexto}-{str(mes_contexto).zfill(2)}-{dia}"

# --- PROCESO PRINCIPAL ---

def ingesta_incremental_final(anio_inicio=2013):
    ahora = datetime.now()
    
    for anio in range(anio_inicio, ahora.year + 1):
        nombre_csv_anual = f"clima_{anio}_bronce.csv"
        
        # Saltamos años pasados si ya existen, pero el año actual lo re-procesamos siempre
        if os.path.exists(nombre_csv_anual) and anio < ahora.year:
            print(f"⏩ Año {anio} ya existe. Saltando...")
            continue
            
        print(f"\n--- 🔄 PROCESANDO AÑO: {anio} ---")
        lista_anual = []
        mes_inicio = 5 if anio == 2013 else 1
        
        for mes in range(mes_inicio, 13):
            if anio == ahora.year and mes > ahora.month:
                break
                
            nombre_rar = f"Aemet{anio}-{mes:02d}.rar"
            url = f"{BASE_URL}{nombre_rar}"
            path_rar = os.path.join(FOLDER_BRONCE, nombre_rar)
            
            try:
                res = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=15)
                if res.status_code == 200:
                    with open(path_rar, "wb") as f:
                        f.write(res.content)
                    
                    # Descomprimir RAR
                    subprocess.run([UNRAR_EXE, 'e', '-o+', path_rar, FOLDER_EXTRACT], capture_output=True)
                    
                    # Procesar Excels interiores
                    ficheros = [f for f in os.listdir(FOLDER_EXTRACT) if f.endswith(('.xls', '.xlsx'))]
                    for f in ficheros:
                        path_f = os.path.join(FOLDER_EXTRACT, f)
                        fecha_registro = extraer_fecha_del_nombre_archivo(f, anio, mes)
                        
                        try:
                            # 1. Buscar fila de cabecera 'Estación'
                            df_preview = pd.read_excel(path_f, header=None, nrows=10).astype(str)
                            idx_header = 4
                            for i, row in df_preview.iterrows():
                                if "estación" in " ".join(row.values).lower():
                                    idx_header = i
                                    break
                            
                            # 2. Cargar datos reales
                            df_dia = pd.read_excel(path_f, skiprows=idx_header)
                            df_dia.columns = [str(c).replace('\n', ' ').strip() for c in df_dia.columns]
                            
                            # Limpieza de filas vacías y asignación de fecha
                            df_dia = df_dia[df_dia['Estación'].notna()].copy()
                            df_dia['fecha'] = fecha_registro
                            
                            # 3. Aplicar LIMPIAR_VALOR a columnas meteorológicas
                            cols_clima = ['Temperatura máxima (ºC)', 'Temperatura mínima (ºC)', 'Temperatura media (ºC)', 
                                         'Precipitación (mm)', 'Racha (km/h)', 'Velocidad máxima (km/h)']
                            
                            for col in cols_clima:
                                if col in df_dia.columns:
                                    df_dia[col] = df_dia[col].apply(limpiar_valor)
                            
                            lista_anual.append(df_dia)
                        except Exception as e:
                            print(f"⚠️ Error procesando archivo {f}: {e}")
                        
                        if os.path.exists(path_f): os.remove(path_f)
                    
                    if os.path.exists(path_rar): os.remove(path_rar)
                    print(f"✅ Mes {mes:02d}/{anio} completado.")
                else:
                    if anio != ahora.year: print(f"❌ {nombre_rar} no encontrado.")
            
            except Exception as e:
                print(f"💥 Error de conexión: {e}")

        # Guardar CSV del año procesado
        if lista_anual:
            df_final = pd.concat(lista_anual, ignore_index=True)
            # Eliminar columnas fantasma Unnamed
            df_final = df_final.loc[:, ~df_final.columns.str.contains('^Unnamed')]
            # Reordenar para que fecha sea la primera
            cols = ['fecha'] + [c for c in df_final.columns if c != 'fecha']
            df_final[cols].to_csv(nombre_csv_anual, index=False)
            print(f"💾 Guardado: {nombre_csv_anual}")

# Ejecutar el motor
ingesta_incremental_final()


--- 🔄 PROCESANDO AÑO: 2013 ---
✅ Mes 05/2013 completado.
✅ Mes 06/2013 completado.
✅ Mes 07/2013 completado.
✅ Mes 08/2013 completado.
✅ Mes 09/2013 completado.
✅ Mes 10/2013 completado.
✅ Mes 11/2013 completado.
✅ Mes 12/2013 completado.
💾 Guardado: clima_2013_bronce.csv

--- 🔄 PROCESANDO AÑO: 2014 ---
✅ Mes 01/2014 completado.
✅ Mes 02/2014 completado.
✅ Mes 03/2014 completado.
⚠️ Error procesando archivo Aemet2014-04-07.xls: Excel file format cannot be determined, you must specify an engine manually.
✅ Mes 04/2014 completado.
✅ Mes 05/2014 completado.
✅ Mes 06/2014 completado.
✅ Mes 07/2014 completado.
✅ Mes 08/2014 completado.
✅ Mes 09/2014 completado.
✅ Mes 10/2014 completado.
✅ Mes 11/2014 completado.
✅ Mes 12/2014 completado.
💾 Guardado: clima_2014_bronce.csv

--- 🔄 PROCESANDO AÑO: 2015 ---
✅ Mes 01/2015 completado.
✅ Mes 02/2015 completado.
✅ Mes 03/2015 completado.
✅ Mes 04/2015 completado.
✅ Mes 05/2015 completado.
✅ Mes 06/2015 completado.
✅ Mes 07/2015 completado.
✅ Mes 08

/tmp/ipykernel_883/982985613.py:126: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_final = pd.concat(lista_anual, ignore_index=True)


💾 Guardado: clima_2015_bronce.csv

--- 🔄 PROCESANDO AÑO: 2016 ---
✅ Mes 01/2016 completado.
✅ Mes 02/2016 completado.
✅ Mes 03/2016 completado.
✅ Mes 04/2016 completado.
✅ Mes 05/2016 completado.
✅ Mes 06/2016 completado.
✅ Mes 07/2016 completado.
✅ Mes 08/2016 completado.
✅ Mes 09/2016 completado.
✅ Mes 10/2016 completado.
✅ Mes 11/2016 completado.
✅ Mes 12/2016 completado.
💾 Guardado: clima_2016_bronce.csv

--- 🔄 PROCESANDO AÑO: 2017 ---
✅ Mes 01/2017 completado.
✅ Mes 02/2017 completado.
✅ Mes 03/2017 completado.
✅ Mes 04/2017 completado.
✅ Mes 05/2017 completado.
✅ Mes 06/2017 completado.
✅ Mes 07/2017 completado.
✅ Mes 08/2017 completado.
✅ Mes 09/2017 completado.
✅ Mes 10/2017 completado.
✅ Mes 11/2017 completado.
✅ Mes 12/2017 completado.
💾 Guardado: clima_2017_bronce.csv

--- 🔄 PROCESANDO AÑO: 2018 ---
✅ Mes 01/2018 completado.
✅ Mes 02/2018 completado.
✅ Mes 03/2018 completado.
✅ Mes 04/2018 completado.
✅ Mes 05/2018 completado.
✅ Mes 06/2018 completado.
✅ Mes 07/2018 completa

/tmp/ipykernel_883/982985613.py:126: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_final = pd.concat(lista_anual, ignore_index=True)


💾 Guardado: clima_2019_bronce.csv

--- 🔄 PROCESANDO AÑO: 2020 ---
✅ Mes 01/2020 completado.
✅ Mes 02/2020 completado.
✅ Mes 03/2020 completado.
✅ Mes 04/2020 completado.
✅ Mes 05/2020 completado.
✅ Mes 06/2020 completado.
⚠️ Error procesando archivo Aemet2020-07-26.xls: Excel file format cannot be determined, you must specify an engine manually.
✅ Mes 07/2020 completado.
✅ Mes 08/2020 completado.
✅ Mes 09/2020 completado.
✅ Mes 10/2020 completado.
✅ Mes 11/2020 completado.
✅ Mes 12/2020 completado.
💾 Guardado: clima_2020_bronce.csv

--- 🔄 PROCESANDO AÑO: 2021 ---
✅ Mes 01/2021 completado.
✅ Mes 02/2021 completado.
✅ Mes 03/2021 completado.
✅ Mes 04/2021 completado.
✅ Mes 05/2021 completado.
✅ Mes 06/2021 completado.
✅ Mes 07/2021 completado.
✅ Mes 08/2021 completado.
✅ Mes 09/2021 completado.
✅ Mes 10/2021 completado.
✅ Mes 11/2021 completado.
✅ Mes 12/2021 completado.
💾 Guardado: clima_2021_bronce.csv

--- 🔄 PROCESANDO AÑO: 2022 ---
✅ Mes 01/2022 completado.
✅ Mes 02/2022 completado.


In [41]:
import glob

# Configuración de rutas
ARCHIVOS = sorted(glob.glob("clima_*_bronce.csv"))
LISTA_FINAL = []

print("🚀 Iniciando unificación inteligente...")

for f in ARCHIVOS:
    try:
        # 1. Leer el archivo completo como texto para buscar Fecha y Cabecera
        df_raw = pd.read_csv(f, header=None, nrows=10).astype(str)
        fila_texto = df_raw[0].str.cat(sep=" ") # Une las primeras filas en un solo texto
        
        # 2. Extraer la fecha (busca el texto después de "Fecha:")
        fecha = fila_texto.split("Fecha:")[1].split()[0:4] if "Fecha:" in fila_texto else f.split('_')[1]
        fecha_str = " ".join(fecha) if isinstance(fecha, list) else fecha

        # 3. Identificar dónde empieza la tabla (fila de "Estación")
        idx_cabecera = df_raw[df_raw.apply(lambda x: x.str.contains("Estación")).any(axis=1)].index[0]

        # 4. Leer datos reales, limpiar columnas y añadir fecha
        df = pd.read_csv(f, skiprows=idx_cabecera)
        df.columns = df.columns.str.replace('\n', ' ').str.strip()
        
        # Limpieza de filas vacías y columnas fantasma
        df = df[df['Estación'].notna()].loc[:, ~df.columns.str.contains('^Unnamed')]
        df['fecha'] = fecha_str
        
        LISTA_FINAL.append(df)
        print(f"✅ {f} procesado. Fecha: {fecha_str}")

    except Exception as e:
        print(f"❌ Error en {f}: {e}")

# 5. Unificación y guardado
if LISTA_FINAL:
    df_maestro = pd.concat(LISTA_FINAL, ignore_index=True)
    df_maestro['fecha'] = pd.to_datetime(df_maestro['fecha'], errors='coerce')
    
    df_maestro.to_csv("clima_plata_unificado.csv", index=False)
    print(f"\n--- Finalizado: {len(df_maestro)} registros unificados ---")

🚀 Iniciando unificación inteligente...
✅ clima_2013_bronce.csv procesado. Fecha: 2013
✅ clima_2014_bronce.csv procesado. Fecha: 2014
✅ clima_2015_bronce.csv procesado. Fecha: 2015
✅ clima_2016_bronce.csv procesado. Fecha: 2016
✅ clima_2017_bronce.csv procesado. Fecha: 2017
✅ clima_2018_bronce.csv procesado. Fecha: 2018
✅ clima_2019_bronce.csv procesado. Fecha: 2019
✅ clima_2020_bronce.csv procesado. Fecha: 2020
✅ clima_2021_bronce.csv procesado. Fecha: 2021
✅ clima_2022_bronce.csv procesado. Fecha: 2022
✅ clima_2023_bronce.csv procesado. Fecha: 2023
✅ clima_2024_bronce.csv procesado. Fecha: 2024
✅ clima_2025_bronce.csv procesado. Fecha: 2025
✅ clima_2026_bronce.csv procesado. Fecha: 2026

--- Finalizado: 3753816 registros unificados ---


In [ ]:
df_preview = pd.read_csv("clima_plata_unificado.csv")


,fecha,Estación,Provincia,Temperatura máxima (ºC),Temperatura mínima (ºC),Temperatura media (ºC),Racha (km/h),Velocidad máxima (km/h),Precipitación 00-24h (mm),Precipitación 00-06h (mm),Precipitación 06-12h (mm),Precipitación 12-18h (mm),Precipitación 18-24h (mm)
3753791,2026-01-01,"San Bartolome Tirajana, Lomo Pedro Alfonso",Las Palmas,22.8,10.8,16.8,47.0,33.0,0.0,0.0,0.0,0.0,0.0
3753792,2026-01-01,"La Aldea de San Nicolás, Tasarte",Las Palmas,25.9,12.6,19.3,38.0,22.0,0.0,0.0,0.0,0.0,0.0
3753793,2026-01-01,"Mogán, Puerto",Las Palmas,25.1,15.2,20.2,55.0,30.0,0.0,0.0,0.0,0.0,0.0
3753794,2026-01-01,"San Bartolome Tirajana, Las Tirajanas",Las Palmas,20.4,10.7,15.5,31.0,16.0,0.0,0.0,0.0,0.0,0.0
3753795,2026-01-01,"Maspalomas, C. Insular Turismo",Las Palmas,26.2,16.3,21.3,42.0,20.0,0.0,0.0,0.0,0.0,0.0
3753796,2026-01-01,"San Bartolome Tirajana, El Matorral",Las Palmas,24.7,16.3,20.5,50.0,32.0,0.0,0.0,0.0,0.0,0.0
3753797,2026-01-01,Agüimes,Las Palmas,23.8,13.9,18.8,54.0,36.0,0.0,0.0,0.0,0.0,0.0
3753798,2026-01-01,"Telde, Centro Forestal Doramas",Las Palmas,21.9,13.3,17.6,32.0,20.0,0.0,0.0,0.0,0.0,0.0
3753799,2026-01-01,Gran Canaria Aeropuerto,Las Palmas,22.9,16.5,19.7,52.0,38.0,0.0,0.0,0.0,0.0,0.0
3753800,2026-01-01,"Telde, Melenara",Las Palmas,22.9,16.7,19.8,32.0,15.0,0.0,0.0,0.0,0.0,0.0


In [46]:
display(df_preview.head(250))

,fecha,Estación,Provincia,Temperatura máxima (ºC),Temperatura mínima (ºC),Temperatura media (ºC),Racha (km/h),Velocidad máxima (km/h),Precipitación 00-24h (mm),Precipitación 00-06h (mm),Precipitación 06-12h (mm),Precipitación 12-18h (mm),Precipitación 18-24h (mm)
3753566,2026-01-01,Cedrillas,Teruel,12.6,2.5,7.6,65.0,45.0,0.0,0.0,0.0,0.0,0.0
3753567,2026-01-01,Mosqueruela,Teruel,11.9,1.1,6.5,86.0,64.0,0.0,0.0,0.0,0.0,0.0
3753568,2026-01-01,Villafranca del Cid/Vilafranca,Castelló/Castellón,13.6,3.9,8.8,76.0,50.0,0.0,0.0,0.0,0.0,0.0
3753569,2026-01-01,Atzeneta del Maestrat,Castelló/Castellón,22.6,11.4,17.0,75.0,33.0,0.0,0.0,0.0,0.0,0.0
3753570,2026-01-01,Castelló - Almassora,Castelló/Castellón,22.1,13.6,17.9,36.0,18.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3753811,2026-01-01,"El Pinar, La Dehesa",Santa Cruz de Tenerife,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3753812,2026-01-01,"San Andrés, Valverde",Santa Cruz de Tenerife,17.1,7.3,12.2,46.0,31.0,0.0,0.0,0.0,0.0,0.0
3753813,2026-01-01,Valverde,Santa Cruz de Tenerife,17.4,10.9,14.1,41.0,29.0,0.0,0.0,0.0,0.0,0.0
3753814,2026-01-01,Hierro Aeropuerto,Santa Cruz de Tenerife,23.7,17.4,20.6,55.0,43.0,0.0,0.0,0.0,0.0,0.0


In [43]:
import holidays

# Definimos el rango de tu proyecto
fecha_inicio = '2013-05-01'
fecha_fin = '2026-12-31'

# 1. Creamos el rango de fechas
fechas = pd.date_range(start=fecha_inicio, end=fecha_fin)
df_festivos = pd.DataFrame({'fecha': fechas})

# 2. Cargamos los festivos oficiales de España (incluye las CCAA)
# Seleccionamos 'ESP' para España y podemos añadir la provincia (ej: 'CL' para Castilla y León)
festivos_esp = holidays.Spain(years=range(2013, 2027), subdiv='CL')

# 3. Aplicamos la lógica al DataFrame
df_festivos['nombre_festivo'] = df_festivos['fecha'].map(festivos_esp)
df_festivos['es_festivo'] = df_festivos['nombre_festivo'].notna().astype(int)

# 4. Añadimos el tipo de día (Laboral / Fin de semana)
df_festivos['dia_semana'] = df_festivos['fecha'].dt.dayofweek
df_festivos['tipo_dia'] = 'Laboral'
df_festivos.loc[df_festivos['dia_semana'] >= 5, 'tipo_dia'] = 'Fin de Semana'
df_festivos.loc[df_festivos['es_festivo'] == 1, 'tipo_dia'] = 'Festivo'

# Guardamos el archivo "oro" de calendario
df_festivos.to_csv("festivos_espana_real.csv", index=False)

print("✅ Calendario Real generado con la librería 'holidays'")
print(df_festivos[df_festivos['es_festivo'] == 1])
display(df_festivos.head(10))

✅ Calendario Real generado con la librería 'holidays'
          fecha                     nombre_festivo  es_festivo  dia_semana  \
0    2013-05-01                          Labor Day           1           2   
106  2013-08-15                     Assumption Day           1           3   
164  2013-10-12                       National Day           1           5   
184  2013-11-01                    All Saints' Day           1           4   
219  2013-12-06                   Constitution Day           1           4   
...         ...                                ...         ...         ...   
4912 2026-10-12                       National Day           1           0   
4933 2026-11-02   Monday following All Saints' Day           1           0   
4968 2026-12-07  Monday following Constitution Day           1           0   
4969 2026-12-08              Immaculate Conception           1           1   
4986 2026-12-25                      Christmas Day           1           4   

     tipo

,fecha,nombre_festivo,es_festivo,dia_semana,tipo_dia
0,2013-05-01,Labor Day,1,2,Festivo
1,2013-05-02,NaN,0,3,Laboral
2,2013-05-03,NaN,0,4,Laboral
3,2013-05-04,NaN,0,5,Fin de Semana
4,2013-05-05,NaN,0,6,Fin de Semana
5,2013-05-06,NaN,0,0,Laboral
6,2013-05-07,NaN,0,1,Laboral
7,2013-05-08,NaN,0,2,Laboral
8,2013-05-09,NaN,0,3,Laboral
9,2013-05-10,NaN,0,4,Laboral


In [44]:

# 1. Cargamos los dos datasets
df_clima = pd.read_csv("clima_plata_unificado.csv")
df_festivos = pd.read_csv("festivos_espana_real.csv")

# 2. IMPORTANTE: Asegurar que la columna 'fecha' es del mismo tipo en ambos
# Si no lo hacemos, el merge puede fallar o dar resultados vacíos
df_clima['fecha'] = pd.to_datetime(df_clima['fecha'])
df_festivos['fecha'] = pd.to_datetime(df_festivos['fecha'])

# 3. Realizamos la unión (Merge)
# Usamos 'left' para mantener todas las filas de clima y añadir los festivos donde coincidan
df_unificado = pd.merge(df_clima, df_festivos, on='fecha', how='left')

# 4. Verificación de seguridad
# Si algún festivo sale como NaN (porque no estaba en el rango), lo rellenamos como día Laboral
df_unificado['es_festivo'] = df_unificado['es_festivo'].fillna(0).astype(int)
df_unificado['tipo_dia'] = df_unificado['tipo_dia'].fillna('Laboral')

# 5. Guardamos el resultado (Este ya es un dataset de nivel "Plata Avanzada")
df_unificado.to_csv("clima_festivos_consolidado.csv", index=False)

print("✅ Unión completada con éxito.")
print(f"Total de registros finales: {len(df_unificado)}")


✅ Unión completada con éxito.
Total de registros finales: 3753816


In [45]:
display(df_unificado.tail(25))

,fecha,Estación,Provincia,Temperatura máxima (ºC),Temperatura mínima (ºC),Temperatura media (ºC),Racha (km/h),Velocidad máxima (km/h),Precipitación 00-24h (mm),Precipitación 00-06h (mm),Precipitación 06-12h (mm),Precipitación 12-18h (mm),Precipitación 18-24h (mm),nombre_festivo,es_festivo,dia_semana,tipo_dia
3753791,2026-01-01,"San Bartolome Tirajana, Lomo Pedro Alfonso",Las Palmas,22.8,10.8,16.8,47.0,33.0,0.0,0.0,0.0,0.0,0.0,New Year's Day,1,3.0,Festivo
3753792,2026-01-01,"La Aldea de San Nicolás, Tasarte",Las Palmas,25.9,12.6,19.3,38.0,22.0,0.0,0.0,0.0,0.0,0.0,New Year's Day,1,3.0,Festivo
3753793,2026-01-01,"Mogán, Puerto",Las Palmas,25.1,15.2,20.2,55.0,30.0,0.0,0.0,0.0,0.0,0.0,New Year's Day,1,3.0,Festivo
3753794,2026-01-01,"San Bartolome Tirajana, Las Tirajanas",Las Palmas,20.4,10.7,15.5,31.0,16.0,0.0,0.0,0.0,0.0,0.0,New Year's Day,1,3.0,Festivo
3753795,2026-01-01,"Maspalomas, C. Insular Turismo",Las Palmas,26.2,16.3,21.3,42.0,20.0,0.0,0.0,0.0,0.0,0.0,New Year's Day,1,3.0,Festivo
3753796,2026-01-01,"San Bartolome Tirajana, El Matorral",Las Palmas,24.7,16.3,20.5,50.0,32.0,0.0,0.0,0.0,0.0,0.0,New Year's Day,1,3.0,Festivo
3753797,2026-01-01,Agüimes,Las Palmas,23.8,13.9,18.8,54.0,36.0,0.0,0.0,0.0,0.0,0.0,New Year's Day,1,3.0,Festivo
3753798,2026-01-01,"Telde, Centro Forestal Doramas",Las Palmas,21.9,13.3,17.6,32.0,20.0,0.0,0.0,0.0,0.0,0.0,New Year's Day,1,3.0,Festivo
3753799,2026-01-01,Gran Canaria Aeropuerto,Las Palmas,22.9,16.5,19.7,52.0,38.0,0.0,0.0,0.0,0.0,0.0,New Year's Day,1,3.0,Festivo
3753800,2026-01-01,"Telde, Melenara",Las Palmas,22.9,16.7,19.8,32.0,15.0,0.0,0.0,0.0,0.0,0.0,New Year's Day,1,3.0,Festivo
